# War Thunder Dataset BBox Size Analysis

This notebook analyzes bbox-size distribution for:

`data/war_thunder_tank_detection_labelme_pairs`

Goals:
1. Compute reliable bbox stats from LabelMe shapes.
2. Report counts below key area-ratio thresholds.
3. Show visual examples with bbox overlays and area ratios.

All threshold outputs are explicit to avoid ambiguity:
- `0.1` ratio means **10%** of image area
- `0.05` ratio means **5%**
- `0.01` ratio means **1%**
- `0.001` ratio means **0.1%**


In [ ]:
from pathlib import Path
import json
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

plt.style.use('ggplot')
pd.set_option('display.max_rows', 200)


In [ ]:
# Config
dataset_dir = Path('data/war_thunder_tank_detection_labelme_pairs')
assert dataset_dir.exists(), f'Missing: {dataset_dir}'

image_exts = {'.png', '.jpg', '.jpeg', '.gif', '.tif', '.tiff', '.bmp', '.webp'}
json_files = sorted(dataset_dir.glob('*.json'))

print('dataset_dir:', dataset_dir)
print('json_files:', len(json_files))


In [ ]:
def bbox_from_shape(shape):
    points = shape.get('points') or []
    if len(points) < 2:
        return None

    st = shape.get('shape_type', '').lower()
    if st == 'rectangle' and len(points) >= 2:
        x0, y0 = points[0]
        x1, y1 = points[1]
        xmin, xmax = sorted([float(x0), float(x1)])
        ymin, ymax = sorted([float(y0), float(y1)])
    else:
        xs = [float(p[0]) for p in points]
        ys = [float(p[1]) for p in points]
        xmin, xmax = min(xs), max(xs)
        ymin, ymax = min(ys), max(ys)

    w = max(0.0, xmax - xmin)
    h = max(0.0, ymax - ymin)
    return xmin, ymin, xmax, ymax, w, h


In [ ]:
# Build annotation dataframe
rows = []

for jf in json_files:
    d = json.loads(jf.read_text())
    iw = float(d.get('imageWidth') or 0)
    ih = float(d.get('imageHeight') or 0)
    if iw <= 0 or ih <= 0:
        continue

    image_area = iw * ih
    image_path = d.get('imagePath', '')

    for i, shape in enumerate(d.get('shapes', []) or []):
        bb = bbox_from_shape(shape)
        if bb is None:
            continue
        xmin, ymin, xmax, ymax, bw, bh = bb
        ba = bw * bh
        rows.append({
            'json_file': jf.name,
            'image_file': image_path,
            'shape_idx': i,
            'label': shape.get('label', ''),
            'shape_type': shape.get('shape_type', ''),
            'image_width': iw,
            'image_height': ih,
            'image_area': image_area,
            'bbox_xmin': xmin,
            'bbox_ymin': ymin,
            'bbox_xmax': xmax,
            'bbox_ymax': ymax,
            'bbox_w': bw,
            'bbox_h': bh,
            'bbox_area': ba,
            'bbox_area_ratio': ba / image_area,
        })

ann = pd.DataFrame(rows)
assert len(ann) > 0, 'No annotations found.'

print('annotations:', len(ann))
ann.head()


In [ ]:
# Core stats
def q(series, p):
    return float(series.quantile(p))

summary = pd.DataFrame([
    {'metric': 'num_annotations', 'value': int(len(ann))},
    {'metric': 'bbox_w_min', 'value': float(ann['bbox_w'].min())},
    {'metric': 'bbox_w_mean', 'value': float(ann['bbox_w'].mean())},
    {'metric': 'bbox_w_median', 'value': float(ann['bbox_w'].median())},
    {'metric': 'bbox_w_p90', 'value': q(ann['bbox_w'], 0.90)},
    {'metric': 'bbox_w_p95', 'value': q(ann['bbox_w'], 0.95)},
    {'metric': 'bbox_w_max', 'value': float(ann['bbox_w'].max())},

    {'metric': 'bbox_h_min', 'value': float(ann['bbox_h'].min())},
    {'metric': 'bbox_h_mean', 'value': float(ann['bbox_h'].mean())},
    {'metric': 'bbox_h_median', 'value': float(ann['bbox_h'].median())},
    {'metric': 'bbox_h_p90', 'value': q(ann['bbox_h'], 0.90)},
    {'metric': 'bbox_h_p95', 'value': q(ann['bbox_h'], 0.95)},
    {'metric': 'bbox_h_max', 'value': float(ann['bbox_h'].max())},

    {'metric': 'bbox_area_min', 'value': float(ann['bbox_area'].min())},
    {'metric': 'bbox_area_mean', 'value': float(ann['bbox_area'].mean())},
    {'metric': 'bbox_area_median', 'value': float(ann['bbox_area'].median())},
    {'metric': 'bbox_area_p90', 'value': q(ann['bbox_area'], 0.90)},
    {'metric': 'bbox_area_p95', 'value': q(ann['bbox_area'], 0.95)},
    {'metric': 'bbox_area_max', 'value': float(ann['bbox_area'].max())},

    {'metric': 'ratio_min', 'value': float(ann['bbox_area_ratio'].min())},
    {'metric': 'ratio_mean', 'value': float(ann['bbox_area_ratio'].mean())},
    {'metric': 'ratio_median', 'value': float(ann['bbox_area_ratio'].median())},
    {'metric': 'ratio_p90', 'value': q(ann['bbox_area_ratio'], 0.90)},
    {'metric': 'ratio_p95', 'value': q(ann['bbox_area_ratio'], 0.95)},
    {'metric': 'ratio_max', 'value': float(ann['bbox_area_ratio'].max())},
])

summary


In [ ]:
# Explicit threshold counts (unambiguous)
thresholds = [
    (0.10, '10% (ratio < 0.1)'),
    (0.05, '5% (ratio < 0.05)'),
    (0.01, '1% (ratio < 0.01)'),
    (0.001, '0.1% (ratio < 0.001)'),
]

rows = []
for t, name in thresholds:
    c = int((ann['bbox_area_ratio'] < t).sum())
    rows.append({
        'threshold_name': name,
        'threshold_ratio': t,
        'count': c,
        'total': int(len(ann)),
        'pct': 100.0 * c / len(ann),
    })

thr_df = pd.DataFrame(rows)
thr_df


In [ ]:
# COCO area buckets (using bbox area in pixels)
small_thr = 32 * 32
medium_thr = 96 * 96

size_bucket = np.where(
    ann['bbox_area'] < small_thr,
    'small(<32^2)',
    np.where(ann['bbox_area'] < medium_thr, 'medium(32^2-96^2)', 'large(>=96^2)')
)

bucket_df = (
    pd.Series(size_bucket)
    .value_counts()
    .rename_axis('bucket')
    .reset_index(name='count')
)
bucket_df['pct'] = 100.0 * bucket_df['count'] / len(ann)
bucket_df


In [ ]:
# Distribution plots
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

axes[0].hist(ann['bbox_w'], bins=40)
axes[0].set_title('BBox Width (px)')
axes[0].set_xlabel('width')

axes[1].hist(ann['bbox_h'], bins=40)
axes[1].set_title('BBox Height (px)')
axes[1].set_xlabel('height')

axes[2].hist(ann['bbox_area_ratio'], bins=50)
axes[2].set_title('BBox Area Ratio')
axes[2].set_xlabel('bbox_area / image_area')

plt.tight_layout()
plt.show()


In [ ]:
# Visual examples with bbox overlays
# We sample images and draw all bboxes from their JSON.
example_jsons = json_files[:8] if len(json_files) >= 8 else json_files

fig, axes = plt.subplots(len(example_jsons), 1, figsize=(10, 3 * len(example_jsons)))
if len(example_jsons) == 1:
    axes = [axes]

for ax, jf in zip(axes, example_jsons):
    d = json.loads(jf.read_text())
    image_name = d.get('imagePath', '')
    img_path = dataset_dir / image_name

    if img_path.exists():
        img = Image.open(img_path).convert('RGB')
        ax.imshow(img)
    else:
        ax.text(0.5, 0.5, f'Missing image: {image_name}', ha='center', va='center')
        ax.set_axis_off()
        continue

    for shape in d.get('shapes', []) or []:
        bb = bbox_from_shape(shape)
        if bb is None:
            continue
        xmin, ymin, xmax, ymax, bw, bh = bb
        rect = patches.Rectangle((xmin, ymin), bw, bh, linewidth=1.5, edgecolor='lime', facecolor='none')
        ax.add_patch(rect)

    ax.set_title(f"{jf.name} | shapes={len(d.get('shapes', []))}")
    ax.set_axis_off()

plt.tight_layout()
plt.show()


In [ ]:
# Sanity checks against recent reported values (optional)
# You can update these expected values if dataset changes.

calc_total = int(len(ann))
calc_below_5pct = int((ann['bbox_area_ratio'] < 0.05).sum())
calc_below_1pct = int((ann['bbox_area_ratio'] < 0.01).sum())
calc_below_10pct = int((ann['bbox_area_ratio'] < 0.1).sum())
calc_below_0_1pct = int((ann['bbox_area_ratio'] < 0.001).sum())

print('total_annotations:', calc_total)
print('below_10pct_ratio(<0.1):', calc_below_10pct)
print('below_5pct_ratio(<0.05):', calc_below_5pct)
print('below_1pct_ratio(<0.01):', calc_below_1pct)
print('below_0.1pct_ratio(<0.001):', calc_below_0_1pct)


In [ ]:
# Plot 4 batches for each threshold: 0.1, 0.05, 0.01, 0.001
# A "batch" here is a group of images that contain at least one bbox below threshold.

thresholds = [0.1, 0.05, 0.01, 0.001]
batches_per_threshold = 4
batch_size = 4  # images per batch

# Build per-image summary with min bbox area ratio
img_group = (
    ann.groupby('image_file')
    .agg(
        min_ratio=('bbox_area_ratio', 'min'),
        n_boxes=('bbox_area_ratio', 'count')
    )
    .reset_index()
)

# Map image_file -> json path for quick loading
image_to_json = {}
for jf in json_files:
    d = json.loads(jf.read_text())
    image_to_json[d.get('imagePath', '')] = jf


def draw_image_with_filtered_boxes(ax, image_file, threshold):
    jf = image_to_json.get(image_file)
    if jf is None:
        ax.text(0.5, 0.5, f'No JSON for {image_file}', ha='center', va='center')
        ax.set_axis_off()
        return

    d = json.loads(jf.read_text())
    img_path = dataset_dir / image_file
    if not img_path.exists():
        ax.text(0.5, 0.5, f'Missing image: {image_file}', ha='center', va='center')
        ax.set_axis_off()
        return

    img = Image.open(img_path).convert('RGB')
    iw = float(d.get('imageWidth') or img.width)
    ih = float(d.get('imageHeight') or img.height)
    ia = max(1.0, iw * ih)

    ax.imshow(img)
    kept = 0
    for shape in d.get('shapes', []) or []:
        bb = bbox_from_shape(shape)
        if bb is None:
            continue
        xmin, ymin, xmax, ymax, bw, bh = bb
        ratio = (bw * bh) / ia
        if ratio >= threshold:
            continue
        kept += 1
        rect = patches.Rectangle((xmin, ymin), bw, bh, linewidth=1.5, edgecolor='yellow', facecolor='none')
        ax.add_patch(rect)
        ax.text(xmin, max(0, ymin - 3), f'{ratio:.4f}', color='yellow', fontsize=7,
                bbox=dict(facecolor='black', alpha=0.45, pad=1, edgecolor='none'))

    ax.set_title(f'{image_file}\nkept<{threshold}: {kept}', fontsize=8)
    ax.set_axis_off()


for t in thresholds:
    eligible = img_group[img_group['min_ratio'] < t]['image_file'].tolist()
    print(f'threshold<{t}: eligible images = {len(eligible)}')
    if len(eligible) == 0:
        continue

    # deterministic sample for reproducibility
    rng = np.random.default_rng(int(t * 1_000_000) + 42)
    needed = min(len(eligible), batches_per_threshold * batch_size)
    sampled = rng.choice(eligible, size=needed, replace=False).tolist()

    # pad to complete grid if needed
    while len(sampled) < batches_per_threshold * batch_size:
        sampled.append(None)

    fig, axes = plt.subplots(batches_per_threshold, batch_size, figsize=(4 * batch_size, 3 * batches_per_threshold))
    fig.suptitle(f'Images with at least one bbox area_ratio < {t}', fontsize=14)

    for idx, ax in enumerate(axes.reshape(-1)):
        image_file = sampled[idx]
        if image_file is None:
            ax.axis('off')
            continue
        draw_image_with_filtered_boxes(ax, image_file, t)

    # annotate batch rows
    for r in range(batches_per_threshold):
        axes[r, 0].text(-0.12, 0.5, f'Batch {r+1}', transform=axes[r, 0].transAxes,
                        rotation=90, va='center', ha='center', fontsize=10)

    plt.tight_layout()
    plt.show()
